# Comparación de decoders

Se comparan cuatro modelos abiertos de hasta 8000 millones de parámetros. Salamandra tiene prioridad porque está recomendado en el enunciado.

Las sondas salen de artículos reales. Esta prueba mide formato, citas, abstención, tiempo y memoria. No mide exactitud jurídica.

In [ ]:
import sys

if "google.colab" in sys.modules:
    import json
    import os
    import shutil
    import subprocess
    from pathlib import Path
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/ai-week-2026")
    REPO = Path("/content/ai-week-2026")

    def sh(comando):
        print("$", comando, flush=True)
        subprocess.run(comando, shell=True, check=True)

    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    if gpu.returncode != 0:
        raise RuntimeError("No hay GPU. Entorno de ejecución > Cambiar tipo de entorno > GPU T4")
    print(gpu.stdout.strip())
    arquitectura = gpu.stdout.split(",")[1].strip().replace(".", "")

    if not REPO.exists():
        sh(f"unzip -q '{DRIVE}/repo.zip' -d '{REPO}'")
    os.chdir(REPO / "notebooks")
    catalogo_colab = json.loads((REPO / "configs/modelos.json").read_text(encoding="utf-8"))
    commit = catalogo_colab["runtime"]["commit"]


    binario = DRIVE / "runtime" / f"llama-server-{commit[:12]}-sm{arquitectura}"
    if not binario.exists():
        fuente = Path("/content/llama.cpp")
        if not (fuente / "CMakeLists.txt").exists():
            sh(f"git init -q {fuente} && cd {fuente} && git remote add origin https://github.com/ggml-org/llama.cpp.git"
               f" && git fetch -q --depth 1 origin {commit} && git checkout -q FETCH_HEAD")
        sh(f"cd {fuente} && cmake -B build -DGGML_CUDA=ON -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF"
           f" -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES={arquitectura} > cmake.log"
           f" && cmake --build build --target llama-server -j $(nproc) > build.log 2>&1 || (tail -40 build.log; exit 1)")
        binario.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(fuente / "build/bin/llama-server", binario)
    servidor = Path("/content/runtime/llama-server")
    servidor.parent.mkdir(exist_ok=True)
    shutil.copy2(binario, servidor)
    servidor.chmod(0o755)
    os.environ["LLAMA_SERVER"] = str(servidor)
    sh(f"{servidor} --version")


    for modelo in catalogo_colab["decoders"]:
        if modelo["nombre"] == "salamandra-7b-instruct":
            destino = REPO / modelo["ruta"]
            destino.parent.mkdir(parents=True, exist_ok=True)
            for nombre in [destino.name, "conversion.json"]:
                if not (destino.parent / nombre).exists():
                    print("Copiando", nombre, "desde Drive", flush=True)
                    shutil.copy2(DRIVE / "modelos/salamandra-7b-instruct" / nombre, destino.parent / nombre)
            continue
        for archivo in modelo["archivos"]:
            destino = REPO / archivo["ruta"]
            if destino.exists() and destino.stat().st_size == archivo["bytes"]:
                continue
            destino.parent.mkdir(parents=True, exist_ok=True)
            sh(f"wget -q -O '{destino}.part' '{archivo['url']}' && mv '{destino}.part' '{destino}'")
    print("Entorno listo en", REPO)

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
import psutil

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

catalogo = json.loads((raiz / "configs/modelos.json").read_text(encoding="utf-8"))
modelos = pd.DataFrame(catalogo["decoders"])
display(modelos[["nombre", "parametros", "licencia", "contexto_tokens", "recomendado", "cuantizacion"]])

,nombre,parametros,licencia,contexto_tokens,recomendado,cuantizacion
0,salamandra-7b-instruct,7768117248,apache-2.0,8192,True,Q4_K_M
1,qwen3-4b,4022468096,apache-2.0,32768,False,Q4_K_M
2,qwen2.5-7b-instruct,7615616512,apache-2.0,32768,False,Q4_K_M
3,qwen2.5-1.5b-instruct,1543714304,apache-2.0,32768,False,Q4_K_M


## Prueba

Se usa el mismo contexto para todos los modelos. Cada artículo entra completo o se omite. No se corta para hacerlo caber.

El servidor aplica la plantilla oficial del modelo con /apply-template y genera con /completion y json_schema. Se guardan la conversación, el prompt final y la respuesta original.

Cada unidad produce cinco sondas. Se prueban los tres formatos con contexto y las respuestas semiabiertas y abiertas sin contexto. Las opciones de selección múltiple son artículos literales, por eso no se usan como prueba sin fundamento.

Se hacen dos corridas con temperatura 0, semilla 0 y sin reutilizar la caché del prompt. Se compara el texto generado y la salida ensamblada. El contexto es de 4096 tokens y la salida puede usar hasta 512.

Se usa gramática JSON en los cuatro modelos. El formato queda guiado por el esquema y no mide su producción espontánea. La opción sin gramática queda disponible en el script. El esquema local debe contrastarse con el oficial del lunes.

La prueba deja responder al modelo sin contexto para observar su abstención. Sus respuestas se conservan intactas. El CLI de producción fuerza abstención cuando no puede incluir pasajes completos o detecta citas sin respaldo. Conserva el raw y el motivo. Las referencias indeterminadas quedan registradas. La calibración con preguntas oficiales queda pendiente.

In [ ]:
volver_a_medir = True
modelos_a_comparar = modelos["nombre"].tolist()
unidades = 1
max_tokens = 512
contexto = 4096
gramatica = "con"

ruta_ultima = raiz / "data/processed/comparacion_decoders/ultima_ejecucion.json"
anterior = json.loads(ruta_ultima.read_text(encoding="utf-8"))["ruta"] if ruta_ultima.exists() else None
tabla = pd.DataFrame()

if volver_a_medir:
    comando = [sys.executable, "-u", str(raiz / "scripts/comparar_decoders.py"),
               "--modelos", *modelos_a_comparar, "--unidades", str(unidades),
               "--max-tokens", str(max_tokens), "--contexto", str(contexto),
               "--gramatica", gramatica]
    entorno = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}
    print(f"Modelos: {', '.join(modelos_a_comparar)}", flush=True)
    print(f"Sondas por modelo: {unidades * 5}. Corridas: 2. Salida máxima: {max_tokens} tokens.", flush=True)
    print("Se verifican los pesos y se carga un modelo a la vez. El avance aparece al terminar cada respuesta.", flush=True)
    with subprocess.Popen(comando, cwd=raiz, env=entorno, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as proceso:
        try:
            for linea in proceso.stdout:
                print(linea, end="", flush=True)
            codigo = proceso.wait()
        finally:
            if proceso.poll() is None:
                principal = psutil.Process(proceso.pid)
                familia = [principal] + principal.children(recursive=True)
                for propio in familia:
                    try:
                        propio.terminate()
                    except psutil.NoSuchProcess:
                        pass
                _, vivos = psutil.wait_procs(familia, timeout=5)
                for propio in vivos:
                    propio.kill()
        if codigo != 0:
            raise RuntimeError(f"La comparación terminó con código {codigo}")

if not ruta_ultima.exists():
    raise FileNotFoundError("No hay una comparación completa guardada. Ejecutar con volver_a_medir = True.")

ultima = json.loads(ruta_ultima.read_text(encoding="utf-8"))
if volver_a_medir and ultima["ruta"] == anterior:
    raise RuntimeError("No se guardó una corrida nueva. La anterior no se usará como resultado actual.")
carpeta = raiz / ultima["ruta"]
configuracion = json.loads((carpeta / "configuracion.json").read_text(encoding="utf-8"))
estados = pd.read_json(carpeta / "estado_modelos.json")
display(estados)
configuracion_esperada = {"contexto": contexto, "max_tokens": max_tokens,
                         "unidades": unidades, "gramatica": gramatica}
if any(configuracion.get(k) != v for k, v in configuracion_esperada.items()):
    raise ValueError("La última corrida no coincide con los parámetros de esta celda.")
if set(estados["modelo"]) != set(modelos_a_comparar):
    raise ValueError("La última corrida corresponde a otra selección de modelos.")
respuestas_esperadas = unidades * 5 * 2 * (2 if gramatica == "ambas" else 1)
completa = estados["estado"].eq("ejecutado").all()
completa = completa and estados.get("respuestas", pd.Series(dtype=int)).eq(respuestas_esperadas).all()
completa = completa and "respuestas" in estados
if not completa:
    raise RuntimeError(f"Comparación parcial o fallida. Revisar el estado mostrado y los archivos en {carpeta}")
tabla = pd.read_json(carpeta / "resumen.json")
if tabla.empty:
    raise RuntimeError("La corrida no produjo una tabla de resultados. No se mostrarán resultados anteriores.")

Modelos: salamandra-7b-instruct, qwen3-4b, qwen2.5-7b-instruct, qwen2.5-1.5b-instruct
Sondas por modelo: 5. Corridas: 2. Salida máxima: 512 tokens.
Se verifican los pesos y se carga un modelo a la vez. El avance aparece al terminar cada respuesta.
salamandra-7b-instruct | multiple_choice | contexto=True | gramatica=True | corrida=1 | JSON=True
salamandra-7b-instruct | semi_open | contexto=True | gramatica=True | corrida=1 | JSON=True
salamandra-7b-instruct | semi_open | contexto=False | gramatica=True | corrida=1 | JSON=True
salamandra-7b-instruct | open_ended | contexto=True | gramatica=True | corrida=1 | JSON=True
salamandra-7b-instruct | open_ended | contexto=False | gramatica=True | corrida=1 | JSON=True
salamandra-7b-instruct | multiple_choice | contexto=True | gramatica=True | corrida=2 | JSON=True
salamandra-7b-instruct | semi_open | contexto=True | gramatica=True | corrida=2 | JSON=True
salamandra-7b-instruct | semi_open | contexto=False | gramatica=True | corrida=2 | JSON=True

In [ ]:
# Solo en Google Colab: copia las corridas y el reporte a Drive, porque /content se borra al cerrar la sesión.
if "google.colab" in sys.modules:
    import shutil

    resultados_drive = Path("/content/drive/MyDrive/ai-week-2026/resultados")
    shutil.copytree(raiz / "data/processed/comparacion_decoders", resultados_drive / "comparacion_decoders",
                    dirs_exist_ok=True)
    if (raiz / "reports/f_decoders.csv").exists():
        shutil.copy2(raiz / "reports/f_decoders.csv", resultados_drive / "f_decoders.csv")
    print("Resultados copiados a", resultados_drive)

## Resultados de la ejecución

Las tablas leen una sola corrida guardada por el comparador. Cada ejecución completa se guarda en una carpeta nueva. Con volver_a_medir = False se leen los últimos resultados sin cargar modelos.

La memoria es un máximo observado por muestreo. La cifra de GPU incluye el uso gráfico del equipo. El respaldo de citas verifica coincidencia en los pasajes y puede dejar referencias para revisión.

Las citas indeterminadas se cuentan aparte. La proporción solo usa coincidencias explícitas como respaldo y se deja vacía si alguna respuesta no pudo auditarse.

In [ ]:
display(tabla[["modelo", "gramatica", "ejecuciones", "json_validos", "claves_validas",
               "abstenciones_sin_contexto", "casos_sin_contexto",
               "repeticiones_identicas", "pares"]])
display(tabla[["modelo", "citas_detectadas", "citas_respaldadas", "citas_sin_respaldo",
               "citas_indeterminadas", "respuestas_no_auditables", "citas_respaldadas_proporcion"]])
display(tabla[["modelo", "gramatica", "mediana_s", "maximo_s", "dentro_22_s",
               "ram_rss_pico_mib", "vram_total_pico_mib"]])

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for eje, columna, titulo, unidad in [
    (ejes[0], "mediana_s", "Tiempo por respuesta", "Segundos"),
    (ejes[1], "vram_total_pico_mib", "Memoria de GPU observada", "MiB"),
]:
    eje.barh(tabla["modelo"], tabla[columna], color="#6487a3")
    eje.set(title=titulo, xlabel=unidad)
    eje.invert_yaxis()
fig.tight_layout()
plt.show()

## Resultados y decisiones

Análisis pendiente 